In [1]:
import logging

logging.config.fileConfig('logging.ini', defaults={'logfilename': 'scraping.log'})

In [2]:
logger = logging.getLogger("sLogger")

In [ ]:
import pandas as pd
import re
import time
import random
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By

def start_driver():
    # https://habr.com/ru/companies/ruvds/articles/796885/
    options = webdriver.ChromeOptions()
    #options.add_argument('--headless')
    width, height = random.choice([(1366, 768), (1920, 1080), (1024, 768)])
    options.add_argument(f"--window-size={width},{height}")
    options.add_argument("--user-agent=Mozilla/5.0...")

    driver = webdriver.Chrome(options=options)
    return driver


def confirm_adult_age(driver):
    try:
        popup = driver.find_element(By.XPATH, "//div[contains(text(), 'Подтвердите, что вы старше 18 лет')]")
        if popup.is_displayed():
            btn = driver.find_element(By.XPATH, "//button[contains(text(), 'Да, мне есть 18')]")
            logger.info(f'Пройдена проверка на возраст - книга 18+')
            btn.click()
            time.sleep(random.uniform(2, 4))
            return True
    except:
        return False


def get_book_links_from_page(driver, page_num):
    if page_num == 1:
        url = 'https://www.litres.ru/popular/'
    else:
        url = f'https://www.litres.ru/popular/?page={page_num}'

    driver.get(url)
    logger.info(f'Запустили по ссылке на страницу {page_num}')

    time.sleep(random.uniform(2, 3))

    confirm_adult_age(driver)
    for i in range(5):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(2, 4))

    book_links = []
    cards = driver.find_elements(By.CSS_SELECTOR, '.baf48440')

    for card in cards:
        try:
            link = card.find_element(By.CSS_SELECTOR, 'a[href*="/book/"], a[href*="/audiobook/"]')
            href = link.get_attribute('href')
            if href:
                href = href.split('?')[0].split('#')[0]
                book_links.append(href)
        except Exception as e:
            logger.exception(e)
            continue

    return book_links

def parse_book_from_html(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')

    book = {
        'title': '',
        'author': '',
        'isbn': '',
        'pages': '',
        'publication_year': '',
        'rating': '',
        'reviews_count': '',
        'price': '',
        'currency': 'RUB',
        'age_restriction': '',
        'genres': '',
        'publisher': '',
        'formats': ''
    }


    title = soup.find('h1')
    if not title:
        title = soup.find(class_='_8dfa70c8')
    if title:
        book['title'] = title.text.strip()


    author = soup.find(class_='_86af713b')
    if author:
        book['author'] = author.text.strip()

    isbn = soup.find('span', itemprop='isbn')
    if isbn:
        book['isbn'] = isbn.text.strip()

    pages = soup.find(class_='a918461a')
    if pages:
        pages_text = pages.text.strip()
        pages_match = re.search(r'(\d+)', pages_text)
        if pages_match:
            book['pages'] = pages_match.group(1)

    publication_year = soup.find(class_='a918461a')
    if publication_year:
        year_match = re.search(r'(\d{4})', publication_year.text)
        if year_match:
            book['publication_year'] = year_match.group(1)

    price = soup.find(class_='c2861f5a')
    if price:
        price_text = price.text.strip()
        price_match = re.search(r'(\d+[\.,]?\d*)', price_text)
        if price_match:
            book['price'] = price_match.group(1).replace(',', '.')


    rating = soup.find(attrs={'data-testid': 'book-factoids__total-rating'})
    if rating:
        book['rating'] = rating.text.strip().replace(',', '.')


    reviews = soup.find(class_='_63638bfa _79cb6115')
    if reviews:
        reviews_text = reviews.text.strip()
        reviews_match = re.search(r'(\d+)', reviews_text)
        if reviews_match:
            book['reviews_count'] = reviews_match.group(1)


    age = soup.find(class_='_6c4e649e')
    if age:
        age_text = age.text.strip()
        if '18+' in age_text:
            book['age_restriction'] = '18+'
        elif '16+' in age_text:
            book['age_restriction'] = '16+'
        elif '12+' in age_text:
            book['age_restriction'] = '12+'


    genres_wrapper = soup.find(attrs={'data-testid': 'book-genres-and-tags__wrapper'})
    if genres_wrapper:
        genres = genres_wrapper.find_all(class_='aa63d864')
        if genres:
            book['genres'] = ', '.join([g.text.strip() for g in genres[:5]])


    publisher = soup.find(attrs={'data-testid': 'book__characteristicsCopyrightHolder'})
    if publisher:
        book['publisher'] = publisher.text.strip()


    formats_wrapper = soup.find(attrs={'data-testid': 'bookCard__characteristicsFileFormat--wrapper'})
    if formats_wrapper:
        formats = formats_wrapper.find_all(attrs={'role': 'listitem'})
        if formats:
            book['formats'] = ', '.join([f.text.strip() for f in formats])
        else:
            book['formats'] = 'текст'
    else:
        book['formats'] = 'текст'

    logger.info(f'Заскрейпили книгу {title.text.strip()}')
    return book


def scrape_book(driver, url, num):
    driver.get(url)

    time.sleep(random.uniform(1, 3))

    confirm_adult_age(driver)
    driver.execute_script("window.scrollTo(0, 500);")

    if num == 0 or num == 1:
        time.sleep(random.uniform(8, 10))
    else:
        time.sleep(random.uniform(1, 3))

    html = driver.page_source
    book = parse_book_from_html(html)
    book['url'] = url
    return book


def save_current_data(books_data, start_page, end_page):
    filename = f'litres_{start_page}_{end_page}.csv'
    df = pd.DataFrame(books_data)

    df['price_numeric'] = pd.to_numeric(df['price'], errors='coerce')
    df['rating_numeric'] = pd.to_numeric(df['rating'], errors='coerce')
    df['pages_numeric'] = pd.to_numeric(df['pages'], errors='coerce')
    df['reviews_numeric'] = pd.to_numeric(df['reviews_count'], errors='coerce')

    df.to_csv(filename, index=False, encoding='utf-8-sig')

def scrape_pages(start_page, end_page):
    driver = start_driver()
    logger.info(f'Запустили driver')
    
    books_data = []

    for page in range(start_page, end_page + 1):
        page_links = get_book_links_from_page(driver, page)

        for i in range(len(page_links)):
            url = page_links[i]
            book = scrape_book(driver, url, i)
            if book:
                books_data.append(book)

            time.sleep(random.uniform(1, 3))

            save_current_data(books_data, start_page, end_page)

        time.sleep(random.uniform(1, 3))

    driver.quit()

    df = pd.DataFrame(books_data)
    return df


start_page = int(input("start_page="))
end_page = int(input("end_page="))
logger.info(f'Зафиксировали страницы с {start_page} по {end_page}')
df = scrape_pages(start_page, end_page)
logger.info("Программа завершена")

In [6]:
df

,title,author,isbn,pages,publication_year,rating,reviews_count,price,currency,age_restriction,genres,publisher,formats,url
0,Проверка браузера перед переходом на www.litre...,,,,,,,,RUB,,,,текст,https://www.litres.ru/book/anna-dzheyn/po-osko...
1,Проверка браузера перед переходом на www.litre...,,,,,,,,RUB,,,,текст,https://www.litres.ru/book/anna-dzheyn/tvoe-se...
2,Фейерверк на ладони,Ольга Назарова,,430,2026,4.9,139,189,RUB,16+,"Антистресс, Ироничная проза, Книги для души, К...",Правообладатель: Автор,"epub,, fb2,, fb3,, ios.epub,, mobi,, pdf,, txt...",https://www.litres.ru/book/olga-stanislavovna-...
3,Проект «Аве Мария»,Энди Вейер,,21,2021,4.8,2117,529,RUB,16+,"Авантюрные приключения, Близкое будущее, Заруб...",Правообладатель: Аудио-ЛАУ,"m4b,, mp3,, zip",https://www.litres.ru/audiobook/endi-veyer/pro...
4,По осколкам твоего сердца,Анна Джейн,,14,2022,4.8,921,749,RUB,16+,"Young adult, Книги о подростках, Любовь и нена...",Правообладатель: Издательство CLEVER,"m4b,, mp3,, zip",https://www.litres.ru/audiobook/anna-dzheyn/po...
5,Проект «Аве Мария»,Энди Вейер,978-5-17-136008-5,530,2021,4.8,2867,449,RUB,16+,"Авантюрные приключения, Близкое будущее, Заруб...",Правообладатель: Издательство АСТ,"epub,, fb2,, fb3,, ios.epub,, mobi,, pdf,, txt...",https://www.litres.ru/book/endi-veyer/proekt-a...
6,Лисья тень,Татьяна Корсакова,,340,2026,4.9,471,199,RUB,16+,"Загадки прошлого, Мистика, Мистические детекти...",Правообладатель: Автор,"epub,, fb2,, fb3,, ios.epub,, mobi,, pdf,, txt...",https://www.litres.ru/book/tatyana-korsakova/l...
7,Твое сердце будет разбито,Анна Джейн,,14,2022,4.8,503,749,RUB,16+,"Young adult, Литрес Авторы, Любовь и ненависть...",Правообладатель: Издательство CLEVER,"m4b,, mp3,, zip",https://www.litres.ru/audiobook/anna-dzheyn/tv...
8,Афродита,Дария Эссес,,490,2026,5.0,182,349,RUB,18+,"Литрес Авторы, Любовные испытания, Остросюжетн...",Правообладатель: Автор,"epub,, fb2,, fb3,, ios.epub,, mobi,, pdf,, txt...",https://www.litres.ru/book/dariya-esses/afrodi...
9,Девятый,Сергей Лукьяненко,978-5-17-182988-9,281,2025,4.8,2405,479,RUB,16+,"Героическая фантастика, Космическая фантастика...",Правообладатель: Издательство АСТ,"epub,, fb2,, fb3,, ios.epub,, mobi,, pdf,, txt...",https://www.litres.ru/book/sergey-lukyanenko/d...
